<a href="https://colab.research.google.com/github/porekhov/drug_design_2024/blob/main/vina_docking_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title **Install dependences**

%%capture

# =========================
# 1. Install dependencies
# =========================
!pip install -q rdkit meeko vina py3Dmol prody biopython requests pdbfixer mdanalysis gemmi

In [ ]:
#@title **Import modules**

from pathlib import Path
import os
import subprocess
import numpy as np
import requests

from rdkit import Chem
from rdkit.Chem import AllChem
from vina import Vina
import py3Dmol

from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda

In [ ]:
#@title **Download the target structure from PDB**

from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda

# set the PDB code of your protein:
#@markdown Input PDB code of your protein:
PDB_ID = "7REK" #@param {type:"string"}

NATIVE_LIGAND_RESNAME = "R6S" #@param {type:"string"}

# first, we will use PDBFixer to download the protein structure
# using its PDB ID and save it as a pdb file using the openmm package:
fixer = PDBFixer(pdbid=PDB_ID)
PDBFile.writeFile(fixer.topology, fixer.positions, open('input.pdb', 'w'))

# now, we can select and save in separate pdb files the structure of protein
# and the structure of the native ligand using MDAnalysis
# here, you can read about the selection syntax:
# https://docs.mdanalysis.org/stable/documentation_pages/selections.html
# read the pdb file
u = mda.Universe('input.pdb')

# change the selection strings for other protein-ligand complexes
target = u.select_atoms('protein and segid A')
ligand = u.select_atoms('resname ' + NATIVE_LIGAND_RESNAME)

ligand = u.select_atoms('resname ' + NATIVE_LIGAND_RESNAME + ' and segid ' + np.unique(ligand.chainIDs)[0])


# write down two separate files with the target protein and the ligand
target.write("target.pdb")
ligand.write("ligand.pdb")

In [ ]:
#@title **Visualize the target-ligand complex**

# show the protein and the ligand

import py3Dmol

view = py3Dmol.view()
view.removeAllModels()
view.setViewStyle({'style':'outline','color':'black','width':0.1})

view.addModel(open('target.pdb','r').read(), format='pdb')
Prot=view.getModel()
Prot.setStyle({'cartoon':{'arrows':True, 'tubes':True, 'style':'oval', 'color':'white'}})
view.addSurface(py3Dmol.VDW,{'opacity':0.6,'color':'white'})


view.addModel(open('ligand.pdb','r').read(),format='pdb')
ref_m = view.getModel()
ref_m.setStyle({},{'stick':{'colorscheme':'magentaCarbon','radius':0.2}})

view.zoomTo()
view.show()

In [ ]:
#@title **Fix the target structure**

# pdbfixer allows to prepare the protein structure for docking
# E.g., add the missing atoms, remove/fix non-standard atoms/residues

fix = PDBFixer(filename='target.pdb')
# find missing residues
# fix.findMissingResidues()
# # find and replace nonstandard residues
# fix.findNonstandardResidues()
# fix.replaceNonstandardResidues()
# # find and add missing atoms
# fix.findMissingAtoms()
# fix.addMissingAtoms()
# write an output file
PDBFile.writeFile(fix.topology, fix.positions, open('target_fix.pdb', 'w'))

In [ ]:
#@title **Predifine the docking box (region where the ligand should bind)**


# this preliminary step is required to define the center and the size of
# the binding site where the docking will be performed
# here, we define it based on the known ligand from the initial structure

# read the ligand structure using MDAnalysis
u = mda.Universe('ligand.pdb')

# define the geometric center of the ligand and save its X, Y, and Z coords
CenterX = float(u.atoms.center_of_geometry()[0])
CenterY = float(u.atoms.center_of_geometry()[1])
CenterZ = float(u.atoms.center_of_geometry()[2])

# define the box size by expanding 5 angstrom from the ligand in X, Y, and Z
minX = u.atoms.positions[:, 0].min() - 5.0
maxX = u.atoms.positions[:, 0].max() + 5.0
minY = u.atoms.positions[:, 1].min() - 5.0
maxY = u.atoms.positions[:, 1].max() + 5.0
minZ = u.atoms.positions[:, 2].min() - 5.0
maxZ = u.atoms.positions[:, 2].max() + 5.0

# size of the box
SizeX = float(maxX - minX)
SizeY = float(maxY - minY)
SizeZ = float(maxZ - minZ)

# print the results:
center = {'center_x':CenterX,'center_y': CenterY, 'center_z': CenterZ}
size = {'size_x':SizeX,'size_y': SizeY,'size_z': SizeZ}

prep =  {
    "receptor_pdb": 'target_fix.pdb',
    "native_ligand_pdb": 'ligand.pdb',
    "box_center": [CenterX, CenterY, CenterZ],
    "box_size": [SizeX, SizeY, SizeZ]
       }


print("Docking box center:", prep["box_center"])
print("Docking box size:  ", prep["box_size"])

In [ ]:
#@title **Enter the ligand SMILES string**

SMILES = "CCOc1ccc2nc(S(N)(=O)=O)sc2c1"  #@param {type:"string"}

QUERY_NAME = "query_ligand"

In [ ]:
#@title **Prepare receptor PDBQT using Meeko, native and quary ligand SDF and PDBQT**

# =========================
# Prepare receptor PDBQT using Meeko
# =========================
# Meeko receptor preparation CLI uses --read_with_prody/-i,
# -o for output basename, and -p to write PDBQT.

cmd = [
    "mk_prepare_receptor.py",
    "-i", prep["receptor_pdb"],
    "-o", "receptor",
    "-p",
    "--allow_bad_res"
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Meeko usually writes receptor_rigid.pdbqt when using -p.
possible_receptors = ["receptor_rigid.pdbqt", "receptor.pdbqt"]
RECEPTOR_PDBQT = None
for f in possible_receptors:
    if Path(f).exists():
        RECEPTOR_PDBQT = f
        break

if RECEPTOR_PDBQT is None:
    raise FileNotFoundError("Could not find receptor PDBQT output from Meeko.")

print("Prepared receptor:", RECEPTOR_PDBQT)

# =========================
# Prepare ligand from SMILES
# =========================

def smiles_to_3d_sdf(smiles, name, sdf_out):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Cannot parse SMILES: {smiles}")

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 1

    status = AllChem.EmbedMolecule(mol, params)
    if status != 0:
        raise RuntimeError(f"RDKit conformer embedding failed for {name}")

    # RDKit MMFF optimization; falls back to UFF if needed.
    if AllChem.MMFFHasAllMoleculeParams(mol):
        AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
    else:
        AllChem.UFFOptimizeMolecule(mol, maxIters=500)

    mol.SetProp("_Name", name)

    writer = Chem.SDWriter(sdf_out)
    writer.write(mol)
    writer.close()

    return sdf_out


QUERY_SDF = smiles_to_3d_sdf(SMILES, QUERY_NAME, f"{QUERY_NAME}.sdf")
print("Query ligand SDF:", QUERY_SDF)

# =========================
# Prepare native ligand for redocking
# =========================
# Preferred route:
# Download the RCSB Chemical Component ideal SDF using the 3-letter ligand code.
# This usually gives better chemistry/bond orders than ligand atoms extracted from PDB.

def download_rcsb_ligand_sdf(resname, sdf_out):
    resname = resname.upper()
    url = f"https://files.rcsb.org/ligands/download/{resname}_ideal.sdf"
    r = requests.get(url, timeout=20)

    if not r.ok or len(r.text.strip()) < 100:
        raise RuntimeError(f"Could not download RCSB ideal SDF for {resname}")

    Path(sdf_out).write_text(r.text)
    return sdf_out


def prepare_native_sdf_from_rcsb_or_pdb(resname, crystal_pdb, sdf_out="native_ligand_for_docking.sdf"):
    try:
        print(f"Trying RCSB Chemical Component SDF for {resname}...")
        tmp_sdf = download_rcsb_ligand_sdf(resname, "native_ligand_ideal_raw.sdf")

        mol = Chem.SDMolSupplier(tmp_sdf, removeHs=False)[0]
        if mol is None:
            raise RuntimeError("RCSB SDF could not be read by RDKit.")

        mol = Chem.AddHs(mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = 2
        AllChem.EmbedMolecule(mol, params)

        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        else:
            AllChem.UFFOptimizeMolecule(mol, maxIters=500)

        mol.SetProp("_Name", f"{resname}_native")

        writer = Chem.SDWriter(sdf_out)
        writer.write(mol)
        writer.close()

        print("Native ligand prepared from RCSB CCD.")
        return sdf_out

    except Exception as e:
        print("RCSB ligand download/preparation failed.")
        print("Falling back to ligand extracted from PDB.")
        print("Reason:", e)

        mol = Chem.MolFromPDBFile(crystal_pdb, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(crystal_pdb, removeHs=False, sanitize=False)
            if mol is None:
                raise RuntimeError("Could not read native ligand from extracted PDB.")
            Chem.SanitizeMol(mol, catchErrors=True)

        mol = Chem.AddHs(mol, addCoords=True)

        if mol.GetNumConformers() == 0:
            params = AllChem.ETKDGv3()
            params.randomSeed = 3
            AllChem.EmbedMolecule(mol, params)

        if AllChem.MMFFHasAllMoleculeParams(mol):
            AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        else:
            AllChem.UFFOptimizeMolecule(mol, maxIters=500)

        mol.SetProp("_Name", f"{resname}_native")

        writer = Chem.SDWriter(sdf_out)
        writer.write(mol)
        writer.close()

        return sdf_out


NATIVE_SDF = prepare_native_sdf_from_rcsb_or_pdb(
    NATIVE_LIGAND_RESNAME,
    prep["native_ligand_pdb"]
)

print("Native ligand SDF:", NATIVE_SDF)

# =========================
# Convert SDF ligands to PDBQT using Meeko
# =========================

def sdf_to_pdbqt(sdf_file, pdbqt_file):
    cmd = [
        "mk_prepare_ligand.py",
        "-i", sdf_file,
        "-o", pdbqt_file
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return pdbqt_file


NATIVE_PDBQT = sdf_to_pdbqt(NATIVE_SDF, "native_ligand.pdbqt")
QUERY_PDBQT = sdf_to_pdbqt(QUERY_SDF, f"{QUERY_NAME}.pdbqt")

print("Native ligand PDBQT:", NATIVE_PDBQT)
print("Query ligand PDBQT: ", QUERY_PDBQT)

In [ ]:
#@title **Run Vina docking (it can take a few minutes)**

# =========================
# Run Vina docking
# =========================

EXHAUSTIVENESS = 16
N_POSES = 5

def run_vina_docking(
    receptor_pdbqt,
    ligand_pdbqt,
    out_pdbqt,
    center,
    box_size,
    exhaustiveness=8,
    n_poses=5
):
    v = Vina(sf_name="vina", cpu=0)

    v.set_receptor(receptor_pdbqt)
    v.set_ligand_from_file(ligand_pdbqt)

    v.compute_vina_maps(
        center=center,
        box_size=box_size
    )

    # Optional: score and locally optimize starting pose
    try:
        score = v.score()
        print(f"Initial score for {ligand_pdbqt}: {score[0]:.3f} kcal/mol")
    except Exception:
        pass

    v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
    energies = v.energies(n_poses=n_poses)

    v.write_poses(out_pdbqt, n_poses=n_poses, overwrite=True)

    return energies


native_energies = run_vina_docking(
    RECEPTOR_PDBQT,
    NATIVE_PDBQT,
    "native_redocked_out.pdbqt",
    prep["box_center"],
    prep["box_size"],
    exhaustiveness=EXHAUSTIVENESS,
    n_poses=N_POSES
)

query_energies = run_vina_docking(
    RECEPTOR_PDBQT,
    QUERY_PDBQT,
    f"{QUERY_NAME}_docked_out.pdbqt",
    prep["box_center"],
    prep["box_size"],
    exhaustiveness=EXHAUSTIVENESS,
    n_poses=N_POSES
)

print("Native redocking energies for up to 5 poses:")
print([float(i[0]) for i in native_energies])

print("Query ligand docking energies for up to 5 poses:")
print([float(i[0]) for i in query_energies])

In [ ]:
#@title **Extract first Vina pose from multi-model PDBQT as normal PDB**

from pathlib import Path

# =========================
# Extract first Vina pose from multi-model PDBQT
# and write it as a normal PDB file
# =========================

def first_pose_pdbqt_to_pdb(pdbqt_file, pdb_file):
    """
    Extract the first pose from a Vina PDBQT file and write it as a PDB file.
    ROOT, BRANCH, TORSDOF and AutoDock-specific charge/type columns are removed.
    """

    lines = Path(pdbqt_file).read_text().splitlines()

    pdb_lines = []
    in_first_model = False
    found_model = False
    finished_first_model = False

    for line in lines:
        record = line[:6].strip()

        if record == "MODEL":
            found_model = True
            if not in_first_model and not finished_first_model:
                in_first_model = True
            continue

        if record == "ENDMDL":
            if in_first_model:
                finished_first_model = True
                break
            continue

        # If there are MODEL records, only read the first MODEL block.
        # If there are no MODEL records, read all ATOM/HETATM lines.
        if found_model and not in_first_model:
            continue

        if record in {"ATOM", "HETATM"}:
            # PDBQT has extra columns after the PDB coordinate section:
            # partial charge and AutoDock atom type.
            # Keep only the standard PDB part.
            pdb_line = line[:66].rstrip()

            # Try to infer element from the PDBQT AutoDock atom type
            # and place it into the PDB element column.
            autodock_type = line[77:].strip().split()[0] if len(line) > 77 and line[77:].strip() else ""
            atom_name = line[12:16].strip()

            if autodock_type:
                element = autodock_type[0].upper()
            else:
                element = ''.join([c for c in atom_name if c.isalpha()])[:1].upper()

            pdb_line = pdb_line.ljust(76) + element.rjust(2)

            pdb_lines.append(pdb_line)

        elif record == "TER":
            pdb_lines.append("TER")

    pdb_lines.append("END")

    Path(pdb_file).write_text("\n".join(pdb_lines) + "\n")

    return pdb_file


native_best = first_pose_pdbqt_to_pdb(
    "native_redocked_out.pdbqt",
    "native_redocked_best.pdb"
)

query_best = first_pose_pdbqt_to_pdb(
    f"{QUERY_NAME}_docked_out.pdbqt",
    f"{QUERY_NAME}_docked_best.pdb"
)

print(native_best)
print(query_best)

In [ ]:
#@title **Visualize the best Vina pose for the redocking of the native ligand**

# =========================
# Print compact summary
# =========================

print("Best native redocking score, kcal/mol:", native_energies[0][0])
print("Best query ligand docking score, kcal/mol:", query_energies[0][0])

print("\nColor legend:")
print("cyan    = native ligand crystal pose")
print("magenta   = redocked native ligand")
print("green = query ligand docked from SMILES")
print("gray    = receptor")

# =========================
# Visualize receptor, crystal native ligand,
# redocked native ligand, and query ligand
# =========================

def read_file(path):
    return Path(path).read_text()

view = py3Dmol.view(width=950, height=700)

# Protein
view.addModel(read_file(prep["receptor_pdb"]), "pdb")
view.setStyle(
    {"model": 0},
    {"cartoon": {"color": "lightgray", "opacity": 0.75}}
)

# Crystal native ligand
view.addModel(read_file(prep["native_ligand_pdb"]), "pdb")
view.setStyle(
    {"model": 1},
    {"stick": {"colorscheme": "cyanCarbon", "radius": 0.15}}
)

# Redocked native ligand
view.addModel(read_file(native_best), "pdb")
view.setStyle(
    {"model": 2},
    {"stick": {"colorscheme": "magentaCarbon", "radius": 0.15}}
)

# Query ligand docked pose
view.addModel(read_file(query_best), "pdbqt")
view.setStyle(
    {"model": 3},
    {"stick": {"colorscheme": "greenCarbon", "radius": 0.15}}
)

view.zoomTo()
view.show()